# Judging and preference pairs

Use standard_factoid or all_factoid_negatives for factoids; whole_response_list and gold_response_list are specifically for list questions.

judge_sample20/judge_sample10 require bank and a question-level prepared questions file in matching question order. Change expected_samples for other bank sizes. source_model overrides the label; otherwise candidate provenance is used. max_new_judge_calls limits new judgments. previous_judgments can point to an earlier judgments directory to reuse its response cache in a fresh run.

three_stage_curriculum takes pair_file and routes C3>C1, C3>C2 and C2>C1 into concept, format and ranking stages. Unexpected directions fail. stage2_ties takes a complete staged input_root. stage2_alignment scores filtering potential from a source_staged_root and Stage 1 run_root.

Execution is **off by default**. Preview needs only Python and the repository. Select a kernel with the project's runtime dependencies before executing. Each method uses that kernel's Python in a separate worker process.


In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "src/notebook_workflows").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.notebook_workflows import describe, plan, execute
from src.notebook_workflows.presets import PRESETS, configuration
print("Project:", PROJECT_ROOT)
print("Python:", sys.executable)


## Choose a preset

- **judge_sample20**: judge_candidates
- **judge_sample10**: judge_candidates
- **standard_factoid**: wrong_entity_pairs
- **all_factoid_negatives**: wrong_entity_pairs
- **extractive_spans**: span_pairs
- **three_stage_curriculum**: curriculum_split
- **stage2_ties**: tie_dataset
- **stage2_alignment**: alignment_filter
- **whole_response_list**: whole_response_pairs
- **gold_response_list**: gold_response_pairs
- **filter**: filter_pairs

Edit the configuration, then rerun the remaining cells. OVERRIDES uses the option names printed by describe(), with underscores. For CLI flags, True includes the named flag and False omits it. Missing inputs block execution.


In [ ]:
CATEGORY = 'pairs'
PRESET = 'standard_factoid'
OVERRIDES = {
    # Set input/model paths here using the fields printed below.
}
RUN = False
ALLOW_API = False

# Optional sequential comparisons: shared OVERRIDES plus per-variant changes.
VARIANTS = [{"label": "default", "overrides": {}}]

METHOD, PARAMETERS = configuration(CATEGORY, PRESET, OVERRIDES)
print(json.dumps({"method": METHOD, "parameters": PARAMETERS}, indent=2))


## Inspect and preview

Options are read from source without importing the training stack. Unknown options and invalid choices are rejected. Outputs go under Artifacts/notebook_runs and cannot overwrite earlier runs. Rerun this cell to allocate fresh plans.


In [ ]:
print(json.dumps(describe(METHOD), indent=2, default=str))
PLANS = []
for variant in VARIANTS:
    method, parameters = configuration(CATEGORY, PRESET, {**OVERRIDES, **variant["overrides"]})
    preview = plan(method, parameters, project_root=PROJECT_ROOT)
    PLANS.append(preview)
    print("\nVariant:", variant["label"])
    print("Method:", preview["method"])
    print("Output:", preview["run_dir"])
    print("Requires GPU:", preview["gpu"], "| Calls API:", preview["api"])
    print("Unresolved inputs:", preview["missing"] or "none")
    print(json.dumps(preview["parameters"], indent=2))


## Execute reviewed plans

Set RUN=True, then rerun configuration and preview. API methods also need ALLOW_API=True. Plans cannot be changed after preview. Each run retains its configuration, input hashes, status and log. A repeated execution of the same plan refuses to overwrite its directory; create a fresh plan instead.


In [ ]:
RESULTS = []
for preview in PLANS:
    if RUN:
        result = execute(preview, run=True, allow_api=ALLOW_API)
        RESULTS.append(str(result))
    else:
        print("Preview only:", preview["method"], "- set RUN=True to execute.")


## Inspect outputs

Use these explicit paths in the next notebook. Historical commands, plots and outputs are preserved in reproducibility/notebook_archive.zip; notebooks/MIGRATION.md maps old notebooks to these workflows.


In [ ]:
for run in RESULTS:
    path = Path(run)
    print("\nRun:", path)
    print((path / "status.json").read_text(encoding="utf-8"))
    print("Log:", path / "run.log")
    print("Outputs:", [p.name for p in path.iterdir()
                      if p.name not in {"run.log", "plan.json", "input_hashes.json"}])
